In [25]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages,MessagesState
from dotenv import load_dotenv 
from langchain_groq import ChatGroq
from IPython.display import Image,display
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
import os 


In [26]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3.6-27b")


In [27]:
@tool 
def getweather(city:str)->str:
    """get the weather of the city will i will provide to you """
    return f"the weather of the {city} is 78 f"

@tool 
def pincode_db_q(qurey:str)->str:

    """your db master and the expert now you need to arrange the get write the quries """
    return  qurey
tool=[getweather,pincode_db_q]



In [28]:
T_llm=llm.bind_tools(tool)

In [29]:
def bot_node(state:MessagesState):
    # Use T_llm instead of llm so the bot knows about the tools
    response=T_llm.invoke(state["messages"])

    # Return "messages" (plural) to match MessagesState
    return {"messages":[response]} 


In [30]:
tool_node = ToolNode(tool)

In [31]:
workflow=StateGraph(MessagesState)
workflow.add_node("botnode",bot_node)
workflow.add_node("tools",tool_node) # Changed from "tool" to "tools"


In [32]:
workflow.add_edge(START, "botnode")

workflow.add_conditional_edges(
    "botnode",
    tools_condition
)

workflow.add_edge("tools", "botnode")

app = workflow.compile()




In [34]:
from math import exp
try:
    display(Image(app.get_graph().draw_mermaid_png()))

except Exception as e:
    print (e)


Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


In [35]:
app.invoke({"user ":"hello how are you "})

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by NameResolutionError("HTTPSConnection(host=\\\'api.smith.langchain.com\\\', port=443): Failed to resolve \\\'api.smith.langchain.com\\\' ([Errno 11001] getaddrinfo failed)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************fb')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


BadRequestError: Error code: 400 - {'error': {'message': "'messages' : minimum number of items is 1", 'type': 'invalid_request_error'}}